# Orbital Braille — VQC Typehead Demo

Interactive walkthrough of the multi-orb encoder/decoder. See [`GLOSSARY.md`](../../GLOSSARY.md) for terminology.

**Tip:** Low-resolution settings here for speed. For publication figures run `run_demo.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from orbital_braille import (
    OrbitalTypehead, TypeheadConfig, PWaveBMGL, EmergentConstants,
    decode_field, build_stable_font, font_separation,
)

PAYLOAD = "I live in Oregon"
cfg = TypeheadConfig(num_orbs=4, grid_size=32, num_times=16,
                     bmgl=PWaveBMGL(gamma_1=1.5), constants=EmergentConstants())
typehead = OrbitalTypehead(cfg, seed=42)
print(f"Font separation: {font_separation(build_stable_font(4, num_glyphs=16)):.3f} rad")

In [ ]:
encoded = typehead.encode(PAYLOAD)
noisy = typehead.propagate_with_turbulence(encoded)
decoded = decode_field(
    noisy, encoded.intensity_time, typehead.font,
    [o.ell for o in encoded.orbs], bmgl=cfg.bmgl,
    rho=encoded.rho, phi=encoded.phi, pulse_ref=encoded.pulse, t=encoded.t,
)
print(f"Shard fidelity: {decoded.shard_fidelity:.4f}")
print(f"Glyph: index={decoded.glyph_index}  fidelity={decoded.glyph_fidelity:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mid = encoded.field_time.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.angle(encoded.field_time[mid]), cmap="hsv")
axes[0].set_title("Phase (clean)")
axes[1].imshow(encoded.intensity_time[mid], cmap="inferno")
axes[1].set_title("Intensity — OAM + Braille")
axes[2].semilogy(encoded.freqs / 1e9, encoded.spectral_shards + 1e-20)
axes[2].set_title("Spectral shards")
axes[2].set_xlabel("GHz")
plt.tight_layout()
plt.show()